|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 1:</h2>|<h1>The Naive Loop<h1>|
|<h2>Section:</h2>|<h1>The arithmetic<h1>|
|<h2>Lecture:</h2>|<h1><b>The KV cache, in bytes per token<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

# The other thing in memory

The weights are fixed. You pay for them once and every user shares them.

The KV cache is not. It grows with every token, for every sequence, and
nobody shares it. It is the reason you cannot simply raise the batch size
until you reach the ridge point.

In [ ]:
# Llama-3-8B
layers   = 32
kv_heads =  8          # not 32. Grouped-query attention.
head_dim = 128
dtype_b  =  2          # bf16

per_token = 2 * layers * kv_heads * head_dim * dtype_b     # 2 = one K, one V
print(f'{per_token/1024:.0f} KB of KV cache per token')

In [ ]:
for ctx in (512, 2048, 8192, 32768):
  print(f'{ctx:>6} tokens -> {ctx*per_token/1e6:8.1f} MB for ONE sequence')

### Grouped-query attention is not a detail

Use `num_attention_heads` instead of `num_key_value_heads` here and you are
wrong by the group factor, which on this model is 4x. Modern models are
designed around KV pressure, and this is where the design shows up.

In [ ]:
naive = 2 * layers * 32 * head_dim * dtype_b      # if every query head had its own KV
print(f'without GQA: {naive/1024:.0f} KB/token')
print(f'with GQA:    {per_token/1024:.0f} KB/token   ({naive/per_token:.0f}x less)')

# What actually fits

In [ ]:
vram      = 12e9
weights   = 8e9 * 1      # 8B params at fp8
available = vram - weights

print(f'{vram/1e9:.0f} GB card')
print(f'{weights/1e9:.0f} GB of weights')
print(f'{available/1e9:.1f} GB left for KV cache\n')

for ctx in (512, 2048, 8192):
  print(f'at {ctx:>5} tokens of context: {available/(ctx*per_token):5.0f} concurrent sequences')

In [ ]:
contexts = np.array([256,512,1024,2048,4096,8192,16384,32768])
seqs     = available / (contexts * per_token)

plt.figure(figsize=(7,4.5))
plt.plot(contexts, seqs, 'ro-')
plt.axhline(175, color='b', ls=':', label='the ridge point: batch you WANT')
plt.xscale('log', base=2); plt.yscale('log')
plt.xlabel('Context length (tokens)'); plt.ylabel('Concurrent sequences that fit')
plt.title('The batch you can afford, against the batch you wanted')
plt.legend(); plt.grid(alpha=.3)
plt.show()

### Read that gap

The blue line is the batch size the last notebook said you should aim for.
The red line is what memory will actually let you hold.

They cross at a context length somewhere around a thousand tokens, and past
that you are rationing. Every idea in the rest of this course lives in that
gap:

- **continuous batching** keeps the slots you do have busy
- **PagedAttention** stops you wasting 60-80% of them on reservation
- **prefix sharing** lets two sequences hold the same blocks
- **quantization** shrinks both lines at once

And this plot assumed nothing is wasted. Stage 06 measures what a real
contiguous allocator throws away, and the answer is most of it.